# $B^+\to K^+\pi^+\pi^-$ toy fit

A compact non-CP B-decay example using `generate_toy`, `FitSession`, automatic fit fractions and fitted projections.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    FitSession, ToyBackground, enable_x64, generate_toy,
    plot_dalitz, plot_square_dalitz,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


In [ ]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
rho_x = Parameter.coefficient("rho.x", 0.60, bounds=(-2, 2), step=0.02, owner="rho")
rho_y = Parameter.coefficient("rho.y", 0.20, bounds=(-2, 2), step=0.02, owner="rho")
nr_x = Parameter.coefficient("NR.x", -0.35, bounds=(-2, 2), step=0.02, owner="NR")
nr_y = Parameter.coefficient("NR.y", 0.15, bounds=(-2, 2), step=0.02, owner="NR")

model = DecayModel(
    channel,
    [
        Resonance("Kstar892", (0,2), RealImag(1.0, 0.0),
                  mass=0.8958, width=0.0474, spin=1),
        Resonance("rho", (1,2), RealImag(rho_x, rho_y),
                  mass=0.7753, width=0.1491, spin=1),
        NonResonant(RealImag(nr_x, nr_y)),
    ],
    normalization_method="square-dalitz",
    normalization_resolution=220,
    normalization_pair=(0,2),
)
truth = {"rho.x":0.60, "rho.y":0.20, "NR.x":-0.35, "NR.y":0.15}


In [ ]:
data = generate_toy(model, 30_000, parameters=truth, seed=303, pool_size=220_000)
plot_dalitz(data, x="s13", y="s23", title="B+ toy")
plt.show()
plot_square_dalitz(
    data, mother_mass=channel.parent_mass, masses=channel.daughter_masses,
    pair=(0,2), title="B+ Square Dalitz"
)
plt.show()


In [ ]:
session = FitSession(model, data)
result = session.fit(
    {"rho.x":0.35, "rho.y":0.0, "NR.x":-0.10, "NR.y":-0.05},
    simplex=True, ncall=40_000,
)
session.report(result)
session.plot_projection(result, "s13")
plt.show()
session.plot_projection(result, "s23")
plt.show()
